# **Phishing-detektering med offentlig BERT-modell**

**1. Installera bibliotek**

In [ ]:
!pip install transformers torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

**2. Importera nödvändiga komponenter**

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

**3. Skapa en enkel träningsdataset (simulerad data i brist på riktig phishing-data) för BERT-träning.**
Innehåller 50 exempel (25 phishing, 25 legitima) med varierande taktiker

In [ ]:
emails = [
    # ================== PHISHING EXAMPLES ==================
    ("Urgent: Your PayPal account has been locked. Verify now to avoid suspension: http://paypal-security.com", 1),
    ("Amazon Alert: Unusual login detected. Secure your account: http://amazon-account-protection.com", 1),
    ("Microsoft Support: Your license will expire today. Renew now: http://microsoft-license-renewal.net", 1),
    ("Bank of America: Suspicious transaction detected. Confirm your identity: http://bofa-secure-login.com", 1),
    ("Netflix: Payment failed. Update your billing info to avoid service interruption: http://netflix-billing-update.org", 1),
    ("Apple ID Alert: Your account has been accessed from new device. Verify now: http://apple-id-verification.com", 1),
    ("DHL Delivery Problem: We couldn't deliver your package. Update address: http://dhl-package-tracker.com", 1),
    ("Instagram: Your account may be deleted. Verify now: http://instagram-account-confirm.com", 1),
    ("Twitter Security: Unauthorized login attempt. Secure your account: http://twitter-password-reset.net", 1),
    ("eBay Notification: Your seller account requires verification: http://ebay-seller-verification.com", 1),
    ("LinkedIn: Someone viewed your profile. Upgrade to see who: http://linkedin-premium-offer.com", 1),
    ("Dropbox: Your storage is full. Upgrade now: http://dropbox-storage-upgrade.net", 1),
    ("Google Drive: Important document shared with you. View now: http://google-drive-docs.com", 1),
    ("Wells Fargo: Your statement is ready. Click to view: http://wellsfargo-statements.com", 1),
    ("Chase Bank: Important security update required: http://chase-online-banking.com", 1),
    ("UPS Delivery Notification: Your package is on hold. Confirm details: http://ups-delivery-confirm.com", 1),
    ("Walmart eGift Card: You've received $100! Claim now: http://walmart-giftcards.com", 1),
    ("Adobe: Your subscription is about to expire. Renew now: http://adobe-subscription-renewal.net", 1),
    ("Zoom: Your meeting recording is available. Click to view: http://zoom-recordings.com", 1),
    ("Spotify: Your premium membership is pending. Complete verification: http://spotify-account-verify.com", 1),
    ("Airbnb: Your reservation is confirmed. View details: http://airbnb-reservation-details.com", 1),
    ("Best Buy Reward: You've earned $50 credit. Redeem now: http://bestbuy-rewards-program.com", 1),
    ("AT&T: Your bill is overdue. Pay now to avoid service interruption: http://att-bill-payment.com", 1),
    ("Verizon Wireless: Important account notification: http://verizon-account-alert.com", 1),
    ("American Express: Suspicious charge detected. Verify now: http://amex-fraud-protection.com", 1),

    # ================== LEGITIMATE EXAMPLES ==================
    ("Team meeting reminder: Tomorrow at 2 PM in Conference Room B", 0),
    ("Monthly newsletter: April company updates attached", 0),
    ("Your invoice #INV-2023-0456 is ready for payment", 0),
    ("Project deadline update: New timeline attached", 0),
    ("Password changed successfully for your account", 0),
    ("Your recent order #ORD-7832 has been shipped", 0),
    ("HR Announcement: New benefits package details", 0),
    ("IT Maintenance: Scheduled downtime tonight 11PM-1AM", 0),
    ("Welcome to our service! Getting started guide attached", 0),
    ("Your subscription to Tech News Weekly has been activated", 0),
    ("Event invitation: Annual company picnic on June 15", 0),
    ("Your job application has been received", 0),
    ("Document shared with you: Q2 Financial Report", 0),
    ("Your support ticket #ST-4872 has been resolved", 0),
    ("New comment on your document 'Project Proposal'", 0),
    ("Your requested export is ready for download", 0),
    ("Webinar reminder: Cloud Security Best Practices tomorrow", 0),
    ("Your account statement for March 2023 is now available", 0),
    ("Action required: Please complete your training by Friday", 0),
    ("Your recent login from Chrome on Windows", 0),
    ("New voicemail received at 10:15 AM", 0),
    ("Your reimbursement request has been approved", 0),
    ("Survey: Tell us about your recent experience", 0),
    ("Holiday schedule: Office closure dates for December", 0)
]

# Konvertera till DataFrame
df = pd.DataFrame(emails, columns=["text", "label"])

# Visa databastatistik
print(f"Total e-postmeddelanden: {len(df)}")
print(f"Phishing e-postmeddelanden: {sum(df['label'] == 1)}")
print(f"Legitima e-postmeddelanden: {sum(df['label'] == 0)}")

# Konvertera till DataFrame
df = pd.DataFrame(emails, columns=["text", "label"])

Total e-postmeddelanden: 49
Phishing e-postmeddelanden: 25
Legitima e-postmeddelanden: 24


**4. Förbereda tokenizer och modell**

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**5. Förbearbeta data**

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_data = df["text"].apply(lambda x: preprocess_function({"text": x}))

**6. Skapa en enkel tränare** (simulerad träning - i verkligheten behövs mer data)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
)

**7. Skapa en prediktionsfunktion**

In [ ]:
def detect_phishing(email_text):
    inputs = tokenizer(email_text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    phishing_prob = probs[0][1].item()

    if phishing_prob > 0.7:
        return f"🚨 PHISHING (confidence: {phishing_prob:.2%})"
    else:
        return f"✅ LEGITIMATE (confidence: {1-phishing_prob:.2%})"

**8. Testa med nya exempel**

In [ ]:
test_emails = [
    "Click this link to reset your password",
    "The project report is due next Friday",
    "Your Amazon account will be suspended",
    "Team lunch next Wednesday at 12 PM"
]

print("🛡️ Phishing Detection Results 🛡️")
print("="*40)
for i, email in enumerate(test_emails, 1):
    print(f"\nEmail {i}: '{email[:50]}...'")
    print(f"Result: {detect_phishing(email)}")

# 💡 Förbättringsmöjligheter:
# 1. Använd en riktig phishing-dataset från Kaggle eller OpenPhish
# 2. Träna modellen längre med fler epoker
# 3. Lägg till länkanalys som extra funktion
# 4. Implementera en Flask-app för realtidsanalys

🛡️ Phishing Detection Results 🛡️

Email 1: 'Click this link to reset your password...'
Result: ✅ LEGITIMATE (confidence: 47.47%)

Email 2: 'The project report is due next Friday...'
Result: ✅ LEGITIMATE (confidence: 47.35%)

Email 3: 'Your Amazon account will be suspended...'
Result: ✅ LEGITIMATE (confidence: 48.08%)

Email 4: 'Team lunch next Wednesday at 12 PM...'
Result: ✅ LEGITIMATE (confidence: 47.37%)


# **Phishing-detektering med offentlig GPT-2**

**1. Installera nödvändiga bibliotek**

In [ ]:
!pip install transformers torch  # Installerar Hugging Face Transformers och PyTorch


**2. Importera bibliotek**

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer # Importerar GPT-2-modellen och dess tokenizer från Hugging Face
import re

**3. Ladda GPT-2 modell och tokenizer**

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**4. Avancerade phishing-indikatorer med precisionsvikter**

In [ ]:
PHISHING_INDICATORS = [
    (r'account\s*(verification|suspended|locked)', 2.5),
    (r'password\s*(reset|expired|change)', 2.5),
    (r'click\s*(here|link|below|now)', 3.5),
    (r'(urgent|important)\s*(action|response|attention)', 3.0),
    (r'immediate\s*(action|attention)', 3.0),
    (r'verify\s*your\s*(account|identity|details)', 2.5),
    (r'(win|won|prize|reward|free)\s*(offer|gift|money|iphone)', 4.0),
    (r'(limited|special)\s*time\s*offer', 2.0),
    (r'(bank|paypal|amazon|ebay|netflix)\s*account', 2.5),
    (r'(login|credentials|password)\s*(required|needed)', 2.0),
    (r'(http|https)://[^\s]+', 1.5),
    (r'dear\s*(customer|user|member|valued)', 1.0),
    (r'(unusual|suspicious)\s*activity', 2.0),
    (r'security\s*alert', 2.5),
    (r'action\s*required', 2.0),
    (r'your\s*account\s*will\s*be\s*suspended', 3.5),
    (r'(compromised|hacked)\s*account', 4.0),
    (r'important:\s*your', 2.0),
    (r'must\s*(respond|act)', 2.5),
    (r'failure\s*to\s*respond', 2.5)
]

def calculate_phishing_score(text):
    """Beräkna phishing-poäng baserat på indikatorer"""
    score = 0
    text_lower = text.lower()

    for pattern, weight in PHISHING_INDICATORS:
        matches = re.findall(pattern, text_lower)
        score += weight * len(matches)

    # Beräkna konfidensgrad mellan 50-99%
    confidence = 50 + min(score * 10, 49)
    return round(confidence, 2)

**5. Beräkna konfidensgrad mellan 50-99%**

In [ ]:
def classify_email(email_text):
    """Klassificera e-post med tre nivåer av säkerhet"""
    confidence = calculate_phishing_score(email_text)

    if confidence > 75:
        return f"❌ PHISHING (confidence: {confidence:.2f}%)"
    elif confidence > 60:
        return f"⚠️ SUSPICIOUS (confidence: {confidence:.2f}%)"
    else:
        return f"✅ LEGITIMATE (confidence: {confidence:.2f}%)"

**6. Testdata med olika typer av e-postmeddelanden**

In [ ]:
emails = [
    "Click this link to reset your password immediately to avoid account suspension.",
    "The project report is due next Friday, please submit it on time.",
    "Your Amazon account will be suspended unless you verify your details now.",
    "Team lunch next Wednesday at 12 PM in the conference room.",
    "Congratulations! You've won a free iPhone! Click here to claim your prize!",
    "Please find attached the quarterly financial report for your review.",
    "URGENT: Your bank account needs verification. Login now to avoid restrictions.",
    "Meeting reminder: Product launch discussion tomorrow at 10 AM.",
    "Dear customer, we detected unusual activity in your account. Please verify your identity.",
    "Your subscription is about to expire. Renew now to avoid service interruption.",
    "Hello team, please review the attached documents before our meeting tomorrow.",
    "IMPORTANT: Your PayPal account has been compromised. Click immediately to secure it.",
    "Must respond within 24 hours to avoid account closure.",
    "Security alert: New device logged into your account."
]

# Testa och visa resultat
print("📧 Phishing Detector - Resultat\n" + "="*40)
for i, email in enumerate(emails, 1):
    result = classify_email(email)
    print(f"\nEmail {i}: '{email[:50]}{'...' if len(email) > 50 else ''}'")
    print(f"Result: {result}")

# Ytterligare funktionalitet för användarinteraktion
print("\n\n🛠️ Ytterligare funktioner:")
print("- Använd classify_email('din text här') för att testa egna meddelanden")
print("- PHISHING_INDICATORS kan enkelt uppdateras med nya mönster")

📧 Phishing Detector - Resultat

Email 1: 'Click this link to reset your password immediately...'
Result: ✅ LEGITIMATE (confidence: 50.00%)

Email 2: 'The project report is due next Friday, please subm...'
Result: ✅ LEGITIMATE (confidence: 50.00%)

Email 3: 'Your Amazon account will be suspended unless you v...'
Result: ❌ PHISHING (confidence: 99.00%)

Email 4: 'Team lunch next Wednesday at 12 PM in the conferen...'
Result: ✅ LEGITIMATE (confidence: 50.00%)

Email 5: 'Congratulations! You've won a free iPhone! Click h...'
Result: ❌ PHISHING (confidence: 99.00%)

Email 6: 'Please find attached the quarterly financial repor...'
Result: ✅ LEGITIMATE (confidence: 50.00%)

Email 7: 'URGENT: Your bank account needs verification. Logi...'
Result: ⚠️ SUSPICIOUS (confidence: 75.00%)

Email 8: 'Meeting reminder: Product launch discussion tomorr...'
Result: ✅ LEGITIMATE (confidence: 50.00%)

Email 9: 'Dear customer, we detected unusual activity in you...'
Result: ❌ PHISHING (confidence: 99.00%)

E